# Fine-tuning Evo-1 on VLABench (Google Colab)

This notebook provides a complete pipeline for fine-tuning the **Evo-1** Vision-Language-Action model on **VLABench** tasks, optimized for **Google Colab**.

## Pipeline Overview

1. **GPU Check**: Verify Colab GPU availability
2. **Google Drive Mount**: Persist datasets and checkpoints across sessions
3. **System Dependencies**: Install headless rendering libraries
4. **Clone Repositories**: Evo-1, VLABench, LeRobot
5. **Python Dependencies**: Install all required packages
6. **Download Assets**: VLABench simulation objects and scenes
7. **Trajectory Generation**: Generate expert trajectories
8. **Dataset Conversion**: Convert HDF5 to LeRobot v2.1 format
9. **Stage 1 Training**: Train integration module + action expert
10. **Stage 2 Training**: Full-scale training (VLM + action head)

## Requirements

- **Colab Runtime**: GPU (T4, V100, or A100 recommended)
- **Google Drive**: Optional but recommended for persistence

## References

- [VLABench Repository](https://github.com/OpenMOSS/VLABench)
- [Evo-1 Repository](https://github.com/MINT-SJTU/Evo-1)
- [LeRobot Repository](https://github.com/huggingface/lerobot)

In [ ]:
# 1) Check GPU availability
import torch

print('CUDA available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        '❌ No GPU detected!\n'
        'Go to: Runtime > Change runtime type > Hardware accelerator > GPU'
    )

print('GPU:', torch.cuda.get_device_name(0))
mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'VRAM: {mem_gb:.1f} GB')

if mem_gb < 12:
    print('⚠️ Warning: Less than 12GB VRAM. Consider using a larger GPU or reducing batch size.')

## 2. Mount Google Drive (Optional but Recommended)

Mounting Drive allows datasets and checkpoints to persist across Colab sessions.
If you skip this, everything will be stored under `/content` and **lost when the runtime resets**.

In [ ]:
# 2) Mount Google Drive (optional)
from pathlib import Path

DRIVE_ROOT = None

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    print('✅ Drive mounted at /content/drive')
except Exception as e:
    print('⚠️ Drive mount skipped/failed:', e)
    print('   Data will be stored in /content (ephemeral)')
    DRIVE_ROOT = None

## 3. Install System Dependencies

VLABench uses MuJoCo with EGL for headless rendering on Colab.

In [ ]:
# 3) Install system dependencies for headless rendering
!apt-get update -qq
!apt-get install -y -qq git wget unzip ffmpeg \
    libosmesa6-dev patchelf libgl1-mesa-glx libegl1-mesa libgles2-mesa \
    build-essential

print('✅ System dependencies installed')

## 4. Configure Headless Rendering

Set environment variables for MuJoCo EGL rendering (required for Colab's headless environment).

In [ ]:
# 4) Configure headless rendering (Colab has no display)
import os
import warnings

# Force EGL rendering for MuJoCo
os.environ['MUJOCO_GL'] = 'egl'

# Remove DISPLAY to prevent X11 windowing attempts
os.environ.pop('DISPLAY', None)

# Suppress harmless glfw warnings
warnings.filterwarnings('ignore', message=r'.*X11: The DISPLAY environment variable is missing.*')

print('MUJOCO_GL =', os.environ.get('MUJOCO_GL'))
print('DISPLAY =', os.environ.get('DISPLAY'))
print('✅ Headless rendering configured')

## 5. Clone Repositories

Clone Evo-1, VLABench, and LeRobot to the workspace.

In [ ]:
# 5) Clone repositories
import subprocess
import shutil
from pathlib import Path

# Workspace root - use Drive if mounted, otherwise ephemeral /content
if DRIVE_ROOT and DRIVE_ROOT.exists():
    WORKSPACE_ROOT = DRIVE_ROOT / "evo1_finetune"
else:
    WORKSPACE_ROOT = Path("/content/evo1_finetune")

WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

# Repository roots (only public repos needed for fine-tuning)
EVO1_ROOT = WORKSPACE_ROOT / "Evo-1"
VLABENCH_ROOT = WORKSPACE_ROOT / "VLABench"
LEROBOT_ROOT = WORKSPACE_ROOT / "lerobot"
RRT_ROOT = WORKSPACE_ROOT / "rrt-algorithms"  # Required by VLABench for motion planning

# Data directories
HDF5_OUT_DIR = WORKSPACE_ROOT / "hdf5_data"
HF_HOME = WORKSPACE_ROOT / "huggingface"
CKPT_ROOT = WORKSPACE_ROOT / "checkpoints"

for d in [HDF5_OUT_DIR, HF_HOME, CKPT_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

# Clone repositories with retry logic
def clone_or_pull(url: str, dest: Path, branch: str = None):
    # Check if valid git repo exists
    if dest.exists():
        git_dir = dest / ".git"
        if git_dir.exists():
            print(f"✓ {dest.name} already exists")
            return
        else:
            # Directory exists but not a valid git repo (corrupted) - remove it
            print(f"⚠️ {dest.name} exists but is not a git repo, removing...")
            shutil.rmtree(dest)
    
    cmd = ["git", "clone", "--depth", "1"]
    if branch:
        cmd += ["-b", branch]
    cmd += [url, str(dest)]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    # Check if clone succeeded despite hook permission errors (common on Google Drive)
    git_dir = dest / ".git"
    if git_dir.exists():
        # Clone succeeded - remove hooks to prevent future permission issues
        hooks_dir = git_dir / "hooks"
        if hooks_dir.exists():
            shutil.rmtree(hooks_dir)
            hooks_dir.mkdir()  # Recreate empty hooks dir
        print(f"✅ Cloned {dest.name}")
        return
    
    # Clone actually failed
    if result.returncode != 0:
        print(f"❌ Failed to clone {dest.name}")
        print(f"   stderr: {result.stderr}")
        raise RuntimeError(f"git clone failed: {result.stderr}")
    print(f"✅ Cloned {dest.name}")

print("Cloning repositories...")
clone_or_pull("https://github.com/MINT-SJTU/Evo-1.git", EVO1_ROOT)
clone_or_pull("https://github.com/OpenMOSS/VLABench.git", VLABENCH_ROOT)
clone_or_pull("https://github.com/huggingface/lerobot.git", LEROBOT_ROOT, branch="v0.3.2")
clone_or_pull("https://github.com/SZanlongo/rrt-algorithms.git", RRT_ROOT)  # Motion planning for VLABench

print("\n📁 Workspace structure:")
print(f"  WORKSPACE_ROOT: {WORKSPACE_ROOT}")
print(f"  EVO1_ROOT: {EVO1_ROOT}")
print(f"  VLABENCH_ROOT: {VLABENCH_ROOT}")
print(f"  LEROBOT_ROOT: {LEROBOT_ROOT}")
print(f"  RRT_ROOT: {RRT_ROOT}")
print(f"  HDF5_OUT_DIR: {HDF5_OUT_DIR}")
print(f"  CKPT_ROOT: {CKPT_ROOT}")

## 6. Install Python Dependencies

Install all required Python packages for Evo-1, VLABench, and LeRobot.

In [ ]:
# 6) Install Python dependencies
import sys

# ============================================================
# STEP 1: Uninstall conflicting pre-installed packages
# ============================================================
!pip uninstall -y -q jax jaxlib pytensor shap dopamine-rl optax flax chex orbax-checkpoint pymc 2>/dev/null || true

# ============================================================
# STEP 2: Install PyTorch 2.5.1 (latest on cu121 index)
# ============================================================
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu121

# ============================================================
# STEP 3: Install OpenCV compatible with LeRobot
# ============================================================
!pip install -q 'opencv-python-headless>=4.9.0'

# ============================================================
# STEP 4: Core ML/Robotics packages
# ============================================================
!pip install -q transformers accelerate safetensors datasets huggingface_hub \
    einops timm scipy pyyaml tensorboard wandb imageio av

# Flash attention (optional - may fail on some GPUs)
!pip install -q flash_attn 2>/dev/null || echo "⚠️ flash_attn install skipped"

# ============================================================
# STEP 5: MuJoCo & simulation
# ============================================================
!pip install -q 'numpy>=1.24,<2.0'
!pip install -q mujoco dm_control gymnasium==0.29.1

# ============================================================
# STEP 6: Open3D for VLABench (point cloud processing)
# ============================================================
!pip install -q 'open3d>=0.18.0'

# Verify open3d installation
try:
    import open3d as o3d
    print(f"✅ Open3D installed: {o3d.__version__}")
except ImportError:
    print("⚠️ Open3D import failed, trying alternative install...")
    !pip install -q open3d

# ============================================================
# STEP 7: RRT-Algorithms (motion planning for VLABench)
# ============================================================
# Add rrt-algorithms to Python path
if str(RRT_ROOT) not in sys.path:
    sys.path.insert(0, str(RRT_ROOT))
print(f"✅ Added rrt-algorithms to path: {RRT_ROOT}")

# ============================================================
# STEP 8: LeRobot (install with --no-deps to avoid version conflicts)
# ============================================================
!pip install -q -e {LEROBOT_ROOT} --no-deps
# Install LeRobot's essential dependencies manually
# Note: 'av' is the PyPI name for pyav (video encoding/decoding)
!pip install -q draccus av rerun-sdk zarr pillow jsonlines numba

# ============================================================
# STEP 9: VLABench
# ============================================================
!pip install -q -e {VLABENCH_ROOT}

# ============================================================
# STEP 10: Evo-1 (no setup.py - add to path + install requirements)
# ============================================================
EVO1_EVO1_DIR = EVO1_ROOT / "Evo_1"

# Add to Python path
if str(EVO1_EVO1_DIR) not in sys.path:
    sys.path.insert(0, str(EVO1_EVO1_DIR))
if str(EVO1_ROOT) not in sys.path:
    sys.path.insert(0, str(EVO1_ROOT))

# Install Evo-1 requirements (excluding torch to prevent conflicts)
evo1_requirements = EVO1_EVO1_DIR / "requirements.txt"
if evo1_requirements.exists():
    # Filter out torch-related packages to prevent version conflicts
    filtered_reqs = []
    for line in evo1_requirements.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#'):
            pkg_lower = line.lower()
            if not any(x in pkg_lower for x in ['torch', 'torchvision', 'torchaudio']):
                filtered_reqs.append(line)
    
    # Write filtered requirements to temp file
    temp_reqs = WORKSPACE_ROOT / "evo1_filtered_requirements.txt"
    temp_reqs.write_text('\n'.join(filtered_reqs))
    !pip install -q -r {temp_reqs} 2>/dev/null || echo "⚠️ Some Evo-1 deps skipped"

print(f"✅ Added Evo-1 to Python path: {EVO1_ROOT}")

# ============================================================
# STEP 11: Final verification
# ============================================================
import torch
print(f"\n✅ Python dependencies installed")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
print(f"   Note: OpenCV numpy warnings are safe to ignore (we need numpy<2 for MuJoCo)")

## 7. Download VLABench Assets

Download the required assets (objects, scenes) for VLABench simulation.

In [ ]:
# 7) Download VLABench assets
import sys

assets_dir = VLABENCH_ROOT / "VLABench" / "assets"

if (assets_dir / "obj").exists() and (assets_dir / "scenes").exists():
    print("✅ VLABench assets already present")
else:
    print("Downloading VLABench assets...")
    download_script = VLABENCH_ROOT / "scripts" / "download_assets.py"
    if download_script.exists():
        subprocess.run([sys.executable, str(download_script)], cwd=str(VLABENCH_ROOT), check=True)
        print("✅ VLABench assets downloaded")
    else:
        print(f"⚠️ Download script not found: {download_script}")

## 8. Configure Fine-tuning Parameters

Set the tasks, dataset name, and training parameters.

In [ ]:
# 8) Fine-tuning configuration

# Tasks to fine-tune on (see VLABench documentation for full list)
TASKS = [
    "select_toy",
    "select_fruit",
    "select_drink",
]

# Dataset configuration
DATASET_NAME = "vlabench_finetune"  # Name for the LeRobot dataset
N_SAMPLES_PER_TASK = 10  # Number of trajectories per task (start small, increase later)
ROBOT = "franka"  # Robot type

# Training configuration
VLM_NAME = "OpenGVLab/InternVL3-1B"  # Vision-Language Model
BATCH_SIZE = 8  # Reduce if OOM
LEARNING_RATE = 1e-5
IMAGE_SIZE = 448
HORIZON = 50

# Stage 1: Train integration module + action expert
STAGE1_MAX_STEPS = 5000

# Stage 2: Full-scale training (VLM + action head)
STAGE2_MAX_STEPS = 80000

print("=" * 50)
print("Fine-tuning Configuration")
print("=" * 50)
print(f"Tasks: {TASKS}")
print(f"Samples per task: {N_SAMPLES_PER_TASK}")
print(f"Dataset name: {DATASET_NAME}")
print(f"VLM: {VLM_NAME}")
print(f"Stage 1 steps: {STAGE1_MAX_STEPS}")
print(f"Stage 2 steps: {STAGE2_MAX_STEPS}")

## 9. Generate Expert Trajectories

Use VLABench's expert policies to generate training trajectories.

**Note**: This step uses MuJoCo simulation and may take some time.

In [ ]:
# 9) Generate trajectories using VLABench's trajectory_generation.py script
trajectory_script = VLABENCH_ROOT / "scripts" / "trajectory_generation.py"

if not trajectory_script.exists():
    raise FileNotFoundError(f"Trajectory generation script not found: {trajectory_script}")

env = os.environ.copy()
env["MUJOCO_GL"] = "egl"

for task in TASKS:
    task_out_dir = HDF5_OUT_DIR / task
    task_out_dir.mkdir(parents=True, exist_ok=True)
    
    # Count existing files to resume from
    existing = len(list(task_out_dir.glob("*.hdf5")))
    
    print(f"\nGenerating trajectories for task: {task}")
    print(f"  Output: {task_out_dir}")
    print(f"  Existing: {existing}, Target: {N_SAMPLES_PER_TASK}")
    
    if existing >= N_SAMPLES_PER_TASK:
        print(f"  ✅ Sufficient trajectories exist, skipping")
        continue
    
    cmd = [
        sys.executable,
        str(trajectory_script),
        "--task-name", task,
        "--save-dir", str(HDF5_OUT_DIR),
        "--n-sample", str(N_SAMPLES_PER_TASK),
        "--start-id", str(existing),
        "--robot", ROBOT,
    ]
    
    subprocess.run(cmd, cwd=str(VLABENCH_ROOT), env=env, check=True)
    print(f"  ✅ Generated trajectories for {task}")

print("\n✅ Trajectory generation complete")

## 10. Convert to LeRobot Dataset Format

Convert the HDF5 trajectories to LeRobot v2.1 dataset format required by Evo-1.

In [ ]:
# 10) Convert HDF5 to LeRobot format
convert_script = VLABENCH_ROOT / "scripts" / "convert_to_lerobot.py"

if not convert_script.exists():
    raise FileNotFoundError(f"Conversion script not found: {convert_script}")

env = os.environ.copy()
env["HF_HOME"] = str(HF_HOME)

cmd = [
    sys.executable,
    str(convert_script),
    "--dataset-name", DATASET_NAME,
    "--dataset-path", str(HDF5_OUT_DIR),
    "--task-list", *TASKS,
    "--max-files", "500",
]

print(f"Converting to LeRobot format...")
print(f"  Dataset name: {DATASET_NAME}")
print(f"  Output: {HF_HOME / 'lerobot' / DATASET_NAME}")

subprocess.run(cmd, cwd=str(VLABENCH_ROOT), env=env, check=True)

LEROBOT_DATASET_PATH = HF_HOME / "lerobot" / DATASET_NAME
print(f"\n✅ LeRobot dataset created: {LEROBOT_DATASET_PATH}")

## 11. Generate Evo-1 Dataset Configuration

Create the YAML configuration file that Evo-1 uses to load the dataset.

In [ ]:
# 11) Evo-1 dataset configuration
import yaml

EVO1_CONFIG_DIR = WORKSPACE_ROOT / "configs"
EVO1_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
EVO1_DATASET_CONFIG = EVO1_CONFIG_DIR / "vlabench_dataset.yaml"

config = {
    "max_action_dim": 24,
    "max_state_dim": 24,
    "max_views": 3,
    "data_groups": {
        "vlabench_franka": {
            DATASET_NAME: {
                "path": str(LEROBOT_DATASET_PATH),
                "view_map": {
                    "image_1": "observation.images.image",
                    "image_2": "observation.images.wrist_image",
                }
            }
        }
    }
}

with open(EVO1_DATASET_CONFIG, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Evo-1 dataset config written: {EVO1_DATASET_CONFIG}")
print("\nConfig content:")
print(EVO1_DATASET_CONFIG.read_text())

## 12. Stage 1 Training

Train the integration module and action expert (frozen VLM).

**Note**: This step requires significant GPU memory and time.

In [ ]:
# 12) Stage 1: Train integration module + action expert
EVO1_EVO1_DIR = EVO1_ROOT / "Evo_1"
STAGE1_SAVE_DIR = CKPT_ROOT / "stage1"
STAGE1_SAVE_DIR.mkdir(parents=True, exist_ok=True)

stage1_cmd = [
    sys.executable, "-m", "accelerate", "launch",
    "--num_processes", "1",
    "--num_machines", "1",
    "scripts/train.py",
    "--run_name", f"Evo1_vlabench_{DATASET_NAME}_stage1",
    "--action_head", "flowmatching",
    "--use_augmentation",
    "--lr", str(LEARNING_RATE),
    "--dropout", "0.2",
    "--weight_decay", "1e-3",
    "--batch_size", str(BATCH_SIZE),
    "--image_size", str(IMAGE_SIZE),
    "--max_steps", str(STAGE1_MAX_STEPS),
    "--log_interval", "10",
    "--ckpt_interval", "2500",
    "--warmup_steps", "1000",
    "--grad_clip_norm", "1.0",
    "--num_layers", "8",
    "--horizon", str(HORIZON),
    "--finetune_action_head",
    "--vlm_name", VLM_NAME,
    "--dataset_config_path", str(EVO1_DATASET_CONFIG),
    "--per_action_dim", "24",
    "--state_dim", "24",
    "--save_dir", str(STAGE1_SAVE_DIR),
    "--disable_wandb",
]

print("Starting Stage 1 training...")
print(f"  Save directory: {STAGE1_SAVE_DIR}")
print(f"  Max steps: {STAGE1_MAX_STEPS}")
print()

subprocess.run(stage1_cmd, cwd=str(EVO1_EVO1_DIR), check=True)

print(f"\n✅ Stage 1 training complete")
STAGE1_CHECKPOINT = STAGE1_SAVE_DIR / f"step_{STAGE1_MAX_STEPS}"
print(f"  Checkpoint: {STAGE1_CHECKPOINT}")

## 13. Stage 2 Training

Full-scale training with VLM + action head fine-tuning.

**Note**: This is the most compute-intensive step.

In [ ]:
# 13) Stage 2: Full-scale training
STAGE2_SAVE_DIR = CKPT_ROOT / "stage2"
STAGE2_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Use checkpoint from Stage 1
STAGE1_CHECKPOINT = STAGE1_SAVE_DIR / f"step_{STAGE1_MAX_STEPS}"

stage2_cmd = [
    sys.executable, "-m", "accelerate", "launch",
    "--num_processes", "1",
    "--num_machines", "1",
    "scripts/train.py",
    "--run_name", f"Evo1_vlabench_{DATASET_NAME}_stage2",
    "--action_head", "flowmatching",
    "--use_augmentation",
    "--lr", str(LEARNING_RATE),
    "--dropout", "0.2",
    "--weight_decay", "1e-3",
    "--batch_size", str(BATCH_SIZE),
    "--image_size", str(IMAGE_SIZE),
    "--max_steps", str(STAGE2_MAX_STEPS),
    "--log_interval", "10",
    "--ckpt_interval", "2500",
    "--warmup_steps", "1000",
    "--grad_clip_norm", "1.0",
    "--num_layers", "8",
    "--horizon", str(HORIZON),
    "--finetune_vlm",  # Enable VLM fine-tuning in Stage 2
    "--finetune_action_head",
    "--vlm_name", VLM_NAME,
    "--dataset_config_path", str(EVO1_DATASET_CONFIG),
    "--per_action_dim", "24",
    "--state_dim", "24",
    "--save_dir", str(STAGE2_SAVE_DIR),
    "--resume",
    "--resume_pretrain",
    "--resume_path", str(STAGE1_CHECKPOINT),
    "--disable_wandb",
]

print("Starting Stage 2 training...")
print(f"  Resume from: {STAGE1_CHECKPOINT}")
print(f"  Save directory: {STAGE2_SAVE_DIR}")
print(f"  Max steps: {STAGE2_MAX_STEPS}")
print()

subprocess.run(stage2_cmd, cwd=str(EVO1_EVO1_DIR), check=True)

print(f"\n✅ Stage 2 training complete")
FINAL_CHECKPOINT = STAGE2_SAVE_DIR / f"step_{STAGE2_MAX_STEPS}"
print(f"  Final checkpoint: {FINAL_CHECKPOINT}")

## 14. Evaluation

Evaluate the fine-tuned model on VLABench tasks.

In [ ]:
# 14) Evaluate the fine-tuned model
eval_script = EVO1_EVO1_DIR / "scripts" / "evaluate.py"

if eval_script.exists():
    FINAL_CHECKPOINT = STAGE2_SAVE_DIR / f"step_{STAGE2_MAX_STEPS}"
    
    for task in TASKS:
        print(f"\nEvaluating on task: {task}")
        
        eval_cmd = [
            sys.executable,
            str(eval_script),
            "--checkpoint", str(FINAL_CHECKPOINT),
            "--task", task,
            "--num_episodes", "10",
        ]
        
        try:
            subprocess.run(eval_cmd, cwd=str(EVO1_EVO1_DIR), check=True)
        except subprocess.CalledProcessError as e:
            print(f"  ⚠️ Evaluation failed: {e}")
else:
    print("⚠️ Evaluation script not found. Please check Evo-1 documentation for evaluation instructions.")

## 🎉 Summary

This notebook walked through the complete Evo-1 fine-tuning pipeline on VLABench for Google Colab:

| Step | Description |
|------|-------------|
| 1 | GPU check |
| 2 | Google Drive mount |
| 3 | System dependencies |
| 4 | Headless rendering |
| 5 | Clone repositories |
| 6 | Python dependencies |
| 7 | VLABench assets |
| 8 | Configure parameters |
| 9 | Generate trajectories |
| 10 | Convert to LeRobot |
| 11 | Generate config |
| 12 | Stage 1 training |
| 13 | Stage 2 training |
| 14 | Evaluation |

### Scaling for Production

| Parameter | Sanity Check | Production |
|-----------|--------------|------------|
| `N_SAMPLES_PER_TASK` | 2-5 | 50-100+ |
| `STAGE1_MAX_STEPS` | 100-500 | 5,000 |
| `STAGE2_MAX_STEPS` | 500-1000 | 80,000 |
| `BATCH_SIZE` | 4-8 | 16-32 |

### Resources

- [Evo-1 Paper](https://arxiv.org/abs/2503.00125)
- [VLABench Paper](https://arxiv.org/abs/2412.00258)
- [LeRobot Documentation](https://huggingface.co/docs/lerobot)

In [ ]:
# Print final summary
print('=' * 60)
print('🎉 Fine-tuning Pipeline Complete!')
print('=' * 60)
print()
print(f'Workspace: {WORKSPACE_ROOT}')
print(f'Dataset: {LEROBOT_DATASET_PATH}')
print(f'Stage 1 checkpoint: {STAGE1_SAVE_DIR}')
print(f'Stage 2 checkpoint: {STAGE2_SAVE_DIR}')
print()
print(f'Tasks trained: {TASKS}')
print(f'Samples per task: {N_SAMPLES_PER_TASK}')
print()
if DRIVE_ROOT and DRIVE_ROOT.exists():
    print('✅ All data saved to Google Drive (persistent)')
else:
    print('⚠️ Data in /content (will be lost on runtime reset)')